In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.imputation import imputation_normal_distribution, log2
from src.correlation import pairwise_correlation
from src.data_processing import pre_processing, calculate_data_completeness, filter_data, summarize_filtered_data
from src.data_processing import compute_pca,generate_pca_plot, calculate_cv
from src.statistical_testing import perform_linear_regression
import pingouin as pg
from matplotlib_venn import venn3, venn3_circles
from venn import venn
import statsmodels.stats.multitest as multi
from tqdm import tqdm
from scipy.stats import pearsonr
from scipy.stats import zscore
import pickle

#### Define directories

In [ ]:
import os
from pathlib import Path

# Get the number of available CPUs
CPUS = os.cpu_count()

# Define the paths for the raw and processed data folders
DATA_FOLDER_ALD = '/Volumes/auditgroupdirs/SUND-CPR-TARGET_PROTEOMICS/ald_replication'
DATA_FOLDER_RAW = Path(os.path.join(DATA_FOLDER_ALD, 'data/raw'))
DATA_FOLDER_PROCESSED = Path(os.path.join(DATA_FOLDER_ALD, 'data/processed'))
DATA_FOLDER_CLINIC = os.path.join(DATA_FOLDER_ALD, 'data/processed')
pQTL_2k_FOLDER = Path('/Volumes/auditgroupdirs/SUND-CPR-TARGET_PROTEOMICS/2k_discovery/pQTL/')

# Ensure base folders are created and define subfolder paths
os.makedirs(DATA_FOLDER_PROCESSED, exist_ok=True)
subfolders = ['tables', 'results', 'figures', 'pQTL', 'dash', 'annotations']
folders = {f: Path(DATA_FOLDER_ALD, f) for f in subfolders}

# Create subfolders if they don't exist
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)
    
Path(folders['pQTL'] / 'gemma').mkdir(exist_ok=True)
gemma_path = Path(folders['pQTL'], 'gemma')

#### Import data and format column headers

In [ ]:
# Read annotation file
annotation_file = pd.read_csv(DATA_FOLDER_RAW / 'Experiment annotation file_upgrade.csv',index_col=[0], sep=';')
annotation_file = annotation_file[annotation_file['Sample type']=='Plasma']
IDmapping_sampleID_to_Batch = dict(zip(annotation_file['Sample ID'], annotation_file['Plate']))
# Get the sample IDs to use for the analysis
non_qa_samples = annotation_file[annotation_file['Group2'] != 'QC']
sample_ids = non_qa_samples['Sample ID'].tolist()

In [ ]:
report_filename = '20231103_Report_Protein Lili long (Normal).tsv'
report_filename = '20240302_110019_Protein Lili long (Normal)_ALD_canonical.tsv'
cols_to_keep = ['R.FileName', 'PG.Genes', 'PG.ProteinAccessions', 'PG.Quantity']

RE_READ = True
if not RE_READ:
    chunk_size = 10000
    file_to_read = pd.read_csv(DATA_FOLDER_PROCESSED / 'data_raw_long.csv', chunksize=chunk_size)
    chunks = []
    for chunk in tqdm(file_to_read):
        chunks.append(chunk)
    report_plasma = pd.concat(chunks)
    data_raw_long = report_plasma.dropna().reset_index().drop(['index'], axis=1)
    
else:   
    chunk_size = 10000
    file_to_read = pd.read_csv(os.path.join(DATA_FOLDER_RAW, report_filename), 
                               delimiter='\t', na_values='Filtered', usecols=cols_to_keep, chunksize=chunk_size)
    chunks = []
    for chunk in tqdm(file_to_read):
        chunks.append(chunk)
    report_plasma = pd.concat(chunks)
    data_raw_long = report_plasma.dropna().reset_index().drop(['index'], axis=1)
    data_raw_long.to_csv(DATA_FOLDER_PROCESSED / 'data_raw_long.csv', index=False)

In [ ]:
data_raw_long = pre_processing(data_raw_long)
protein_ids = data_raw_long[['Gene names', 'Protein IDs', 
                             'Gene name', 'Protein ID',
                             'ProteinID_Genename']].drop_duplicates()

In [ ]:
cond1 = data_raw_long['Sample ID']=='Plate1_1'
cond2 = data_raw_long['Protein ID']=='Q9Y6Z7'
test_value=data_raw_long[(cond1) & (cond2)]['PG.Quantity'].iloc[0]
assert abs(test_value-387.400848)<1e-4, 'Value changed compared to previous run'

In [ ]:
#Prepare ID list for genome coordinate mapping
protein_ids['Protein IDs_list'] = protein_ids['Protein IDs'].str.split(';')
protein_ids_all = protein_ids.explode('Protein IDs_list')

with open(gemma_path / 'protein_id_all.list', 'w') as file:
    for protein in protein_ids_all['Protein IDs_list']:
        file.write(str(protein) + '\n')
        
protein_ids_all.to_csv(gemma_path / 'protein_id_all.txt', index=False, sep='\t')

In [ ]:
cols_to_keep = ['Sample ID', 'ProteinID_Genename', 'PG.Quantity']
data_plasma_raw = data_raw_long[cols_to_keep].pivot(columns='Sample ID', index='ProteinID_Genename', values='PG.Quantity')

In [ ]:
df_raw_comp = calculate_data_completeness(data_plasma_raw)
sns.lineplot(x='rank', y='%Complete', data=df_raw_comp)

In [ ]:
df_sig_primary_discovery = pd.read_csv(os.path.join(pQTL_2k_FOLDER, 'df_sig_primary_final.csv'))
len(set(df_sig_primary_discovery['phenotype']) & set(data_plasma_raw.index))

#### Filter data based on data completeness

In [ ]:
proteins, sample_ids, data_plasma_filtered = filter_data(data_plasma_raw)

#### Summarize filtered data

In [ ]:
summarize_filtered_data(data_plasma_filtered)

#### Plot proteins by data completeness

In [ ]:
df_filtered_comp = calculate_data_completeness(data_plasma_filtered)
sns.lineplot(x='rank', y='%Complete', data=df_filtered_comp)
plt.ylim(0, 1.1)

In [ ]:
data_plasma_filtered_log = data_plasma_filtered.apply(np.log2)

#### Import clinical data

In [ ]:
df_cli = pd.read_csv(os.path.join(DATA_FOLDER_PROCESSED,'labtest_integrated_all.csv'))
col_tokeep = ['Sample ID', 'age', 'bmi', 
              'gender_num', 'abstinent_num', 'statin_num', 
              'kleiner', 'nas_inflam', 'nas_steatosis_ordinal', 'biopsy']

In [ ]:
df_cli_pqtl = df_cli[col_tokeep]
for col in ['nas_inflam', 'nas_steatosis_ordinal']:
    indices = df_cli_pqtl[(df_cli_pqtl[col].isnull()) & (df_cli_pqtl['biopsy']=='no')].index
    df_cli_pqtl.loc[indices, col] = 0

In [ ]:
df_cli_pqtl = df_cli_pqtl.drop('biopsy', axis=1).set_index('Sample ID')
for score in ['kleiner', 'nas_inflam', 'nas_steatosis_ordinal']:
    df_cli_pqtl[score] = df_cli_pqtl[score].replace({-1:0, 0.5:0})
    
df_cli_pqtl = df_cli_pqtl.join(annotation_file[['Sample ID_old', 'Sample ID']].set_index('Sample ID_old')).rename_axis('Sample ID_old', axis=0).set_index('Sample ID')

#### Check if there is batch effect by PCA

In [ ]:
IDmapping_proteinid_to_genename = protein_ids[['Protein ID', 'Gene name']].set_index('Protein ID').to_dict()['Gene name']

In [ ]:
df_cli_pqtl.head(3)

In [ ]:
X_train = data_plasma_filtered_log.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]

In [ ]:
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')

In [ ]:
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='Plate')
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

#### Imputation

In [ ]:
data_plasma_filtered_log_imputed = data_plasma_filtered_log.apply(imputation_normal_distribution)
value = data_plasma_filtered_log_imputed.loc['Q9Y6Z7_COLEC10', 'Plate1_2']
assert abs(value - 8.769206) < 0.00001, 'Imputed value changed in comparison to previous run'

In [ ]:
X_train = data_plasma_filtered_log_imputed.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='Plate')
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

#### Normalization

In [ ]:
from combat.pycombat import pycombat
dm = data_plasma_filtered_log_imputed.copy()
batch = [IDmapping_sampleID_to_Batch[i] for i in dm.columns]
data_plasma_filtered_log_imputed_corrected = pycombat(dm, batch)
value = data_plasma_filtered_log_imputed_corrected.loc['Q9Y6Z7_COLEC10', 'Plate1_2']
assert abs(value - 8.798269) < 1e-4, 'Corrected value changed in comparison to previous run'

In [ ]:
X_train = data_plasma_filtered_log_imputed_corrected.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
df_pc = df_pc.join(df_cli_pqtl, how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='Plate')

In [ ]:
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, group_column='kleiner')

#### Save datasets

In [ ]:
FILE_RESULTS = DATA_FOLDER_PROCESSED / 'proteomics_datasets.xlsx'

def format_dataset(df):
    df_new=df.copy()
    df_new.rename_axis('ProteinID_Genename', axis=0, inplace=True)
    cols_to_add = ['Protein ID', 'Protein IDs', 'Gene name', 'Gene names']
    df_new=df_new.join(protein_ids.set_index('ProteinID_Genename')[cols_to_add])
    return df_new

# with pd.ExcelWriter(FILE_RESULTS) as writer:
#     format_dataset(data_plasma_raw).to_excel(writer, sheet_name='raw')
#     format_dataset(data_plasma_filtered_log).to_excel(writer, sheet_name='filtered_log2')
#     format_dataset(data_plasma_filtered_log_imputed).to_excel(writer, sheet_name='filtered_log2_imputed')
#     format_dataset(data_plasma_filtered_log_imputed_corrected).to_excel(writer, sheet_name='imputed_batchcorrected')

### Quality assessment

#### Calculate CV based on 44 quality assessment samples (pooled plasma allocated in 11 plates)

In [ ]:
qa_plasma = annotation_file[annotation_file['Group2'] == 'QC']['Sample ID'].tolist()
df_cv = calculate_cv(data_plasma_filtered, qa_samples=qa_plasma).sort_values(by='Coefficient of variation')
df_cv['Gene name'] = df_cv.index.str.split('_').str[1]

In [ ]:
df_cv.tail(3)

#### Depth

In [ ]:
prot_dep_wide = pd.DataFrame({'raw': data_plasma_raw.count(), 'filtered':data_plasma_filtered.count()})
prot_dep = pd.melt(prot_dep_wide, var_name='dataset', value_name='Number of proteins')
prot_dep.groupby('dataset')['Number of proteins'].median()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12,4))
sns.boxplot(data = prot_dep, x='dataset', y='Number of proteins', color='white', ax=ax1)
ax1.set_ylim(0, 1800)
for i,box in enumerate(ax1.artists):
    box.set_edgecolor('black')
    box.set_facecolor('white')

    # iterate over whiskers and median lines
    for j in range(6*i,6*(i+1)):
         ax1.lines[j].set_color('black')
ax2 = sns.scatterplot(x=df_cv['rank'], y=df_cv['Protein abundance [Log10]'], ax=ax2)
ax3 = sns.scatterplot(x=df_cv['Protein abundance [Log2]'], y=df_cv['Coefficient of variation'], hue=df_cv['color'], ax=ax3)
plt.rcParams['pdf.fonttype'] = 42
#plt.savefig('1k_replication/figures/data_quality.pdf', dpi=120, bbox_inches='tight')

### Combine proteomics and clinical data

In [ ]:
data_combined = df_cli_pqtl.dropna().join(data_plasma_filtered_log_imputed_corrected.T)
proteins = data_plasma_filtered_log_imputed_corrected.index

In [ ]:
mask = data_plasma_filtered_log.isna()
data_reverted = data_plasma_filtered_log_imputed_corrected.mask(mask)
data_combined_noimpute = data_reverted.T.join(df_cli_pqtl).rename_axis('Sample ID', axis=0)
data_combined_noimpute = data_combined_noimpute.join(annotation_file.set_index('Sample ID'), how='left')

#### Normality test before transformation

In [ ]:
# Long data format to ease computation 
data_long = data_combined[proteins].melt(var_name='ProteinID_Genename', value_name='MS signal [Log2]').set_index('ProteinID_Genename')

from src.statistical_testing import normality_pg
# Set a dummy variable needed for pingouin.normality
data_long['group_dummy']=1
normality_results = normality_pg(data=data_long, dv='MS signal [Log2]', group='group_dummy')

# Show results
print('Number of protein with non-normal distribution:{}'.format(normality_results['normal'].value_counts()[False]))

#### Ranked-based inverse normalized transformation (INT)
- https://stackoverflow.com/questions/15549836/transform-data-to-fit-normal-distribution
- [good discussion on violating OLS residual normality assumption](https://stats.stackexchange.com/questions/29731/regression-when-the-ols-residuals-are-not-normally-distributed)
- [good discussion on testing non-linear association](https://stats.stackexchange.com/questions/35893/how-do-i-test-a-nonlinear-association)

In [ ]:
# Perform ranked-based inverse normalized transformation (INT) on protein levels per protein
# Refer to https://github.com/edm1/rank-based-INT

from src.rank_based_int import rank_INT
RE_INT = False

if not RE_INT:
    data_proteomics_int = pd.read_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected_int.csv').set_index('Sample ID')
else:
    new_df = []
    for protein in tqdm(proteins):
        new_df.append(pd.DataFrame(rank_INT(data_plasma_filtered_log_imputed_corrected.T[protein], stochastic=False), columns=[protein]))
    data_proteomics_int = pd.concat(new_df, axis=1)
    data_proteomics_int.rename_axis('Sample ID', axis=0, inplace=True)
    data_proteomics_int.to_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected_int.csv')    

In [ ]:
value = data_proteomics_int.loc['Plate1_2', 'Q9Y6Z7_COLEC10']
assert abs(value - -0.413367) < 1e-6, 'Normalized value changed in comparison to previous run'

In [ ]:
data_combined_int = df_cli_pqtl.dropna().join(data_proteomics_int)

#### Run linear regression correcting for covariates

In [ ]:
covariates_pqtl = df_cli_pqtl.columns.tolist()

In [ ]:
RE_LIREG = True

if not RE_LIREG:
    stats_pqtl = pd.read_csv(gemma_path / 'lireg_statistics.csv')
    residuals_pqtl = pd.read_csv(gemma_path / 'lireg_residuals.csv').set_index('Sample ID')
else:
    stats_pqtl, residuals_pqtl = perform_linear_regression(data_combined_int, proteins, covariates_pqtl)
    # Save results
    stats_pqtl.to_csv(gemma_path / 'lireg_statistics.csv', index=False)
    residuals_pqtl = pd.DataFrame.from_dict(residuals_pqtl).rename_axis('Sample ID', axis=0)
    residuals_pqtl.to_csv(gemma_path / 'lireg_residuals.csv')

In [ ]:
value = residuals_pqtl.loc['Plate1_2', 'Q9Y6Z7_COLEC10']
assert abs(value - -0.185628) < 1e-6, 'Normalized value changed in comparison to previous run'

#### get overlapping IDs between genotype data and proteomics data

In [ ]:
genotype_sampleIDs = pd.read_csv(folders['pQTL'] / 'genomics_sampleIDs.txt', sep='\t')

In [ ]:
df_cli = df_cli.rename({'Sample ID':'Sample ID_old'}, axis=1)
df_cli['Sample ID'] = df_cli['Sample ID_old'].map(dict(zip(annotation_file['Sample ID_old'], annotation_file['Sample ID'])))

In [ ]:
df = df_cli[['Participant ID', 'Sample ID']].copy()
df['cohort'] = df['Participant ID'].str.split('_').str[0].replace('SIPHON', 'ALD')
df['ID'] = df['Participant ID'].str.split('_').str[1]
df['genotype_id'] = df.apply(lambda row: '91x' + row['cohort'] + row['ID'] if row['cohort'] == 'ALD' 
                             else '92x' + row['cohort'] + row['ID'] if row['cohort'] == 'HP' 
                             else row['ID'], axis=1)
sampleID_genotypeID_key = df.copy()

In [ ]:
overlap = list(set(df['genotype_id']) & set(genotype_sampleIDs['Sample ID']))
with open (folders['pQTL'] / 'ordered_sampleID.txt', 'w') as file:
    for i in overlap:
        file.write(i+'\n')

In [ ]:
df_included = df.set_index('genotype_id').loc[overlap]

In [ ]:
RE_INT = True

if not RE_INT:
    data_gwas_int = pd.read_csv(gemma_path / 'lireg_residuals_int.csv').set_index('Sample ID')
else:
    new_df = []
    data = residuals_pqtl
    for protein in tqdm(data.columns):
        new_df.append(pd.DataFrame(rank_INT(data[protein], stochastic=False), columns=[protein]))
    data_gwas_int = pd.concat(new_df, axis=1)
    data_gwas_int.rename_axis('Sample ID', axis=0, inplace=True)
    data_gwas_int.to_csv(gemma_path / 'lireg_residuals_int.csv')

In [ ]:
data_gwas_int['genotypeID'] = data_gwas_int.index.map(dict(zip(df_included['Sample ID'], df_included.index)))
data_gwas_int=data_gwas_int.set_index('genotypeID')

#### Sex check

In [ ]:
plink_sex = pd.read_csv(folders['pQTL'] / 'plink.sexcheck', sep = '\s+')

In [ ]:
df_sexcheck = df_included.join(plink_sex.set_index('FID')).set_index('Sample ID').join(df_cli_pqtl, how='left')

In [ ]:
df_sexcheck['gender_num']=df_sexcheck['gender_num'].replace(0, 2)

In [ ]:
df_sex_mismatch = df_sexcheck[df_sexcheck['SNPSEX']!=df_sexcheck['gender_num']]

In [ ]:
remove_IDs = df_sex_mismatch[['IID']].reset_index().drop('Sample ID', axis=1)
remove_IDs['FID'] = remove_IDs['IID']
remove_IDs=remove_IDs[['FID', 'IID']]

In [ ]:
remove_IDs.to_csv(folders['pQTL'] / 'remove_sexmismatch_ids.txt', index=False, sep=' ')

In [ ]:
remove_IDs

#### import .fam file

In [ ]:
fam_tep = pd.read_csv(folders['pQTL'] / 'ald5.fam', header=None, sep=' ').set_index(0)
data_gwas_export = fam_tep.join(data_gwas_int.round(5), how='left', sort=False).reset_index().drop([5], axis=1).set_index('index')
data_gwas_export = data_gwas_export.reindex(fam_tep.index).reset_index()
final_included_samples = data_gwas_export.dropna()[0]
data_gwas_export.fillna('NA', inplace=True)

In [ ]:
data_gwas_export.to_csv(gemma_path / 'phenotype.fam', header=None, index=False, sep=' ')

In [ ]:
# Check if data changed in comparison to previous run
value = data_gwas_export.loc[0, 'P01009_SERPINA1']
assert abs(value - -0.47993) < 1e-6, 'Exported value for GWAS changed in comparison to previous run'

In [ ]:
# Create a dataframe mapping phenotype IDs to protein IDs
nr_proteins = len(proteins)
proteinID_pqtl=pd.DataFrame({'Phenotype ID':np.arange(1, nr_proteins+1),'Protein ID':data_gwas_export.columns[5:]})
proteinID_pqtl.to_csv(gemma_path / 'ProteinID.txt', index=False, sep='\t')

with open(gemma_path / 'phenotype.list', 'w') as file:
    for line in np.arange(1, nr_proteins+2):
        file.write(str(line) + '\n')
        
# Association performed in Computerome (GEMMA v0.98.3)

#### Export data to look at "dose-response"

In [ ]:
# Define paths
dose_resp_path = folders['pQTL'] / 'dose-response'
dose_resp_path.mkdir(exist_ok=True)
excel_path = dose_resp_path / 'dose_response.xlsx'

IDmapping_SampleID_to_bloodSampleID = dict(zip(sampleID_genotypeID_key['Sample ID'], sampleID_genotypeID_key['genotype_id']))

# Define a function to prepare dataframes
def prepare_df(df):
    df = df.copy()
    df.columns = df.columns.map(IDmapping_SampleID_to_bloodSampleID)
    df = df.rename_axis('Participant ID', axis=1)
    return df

# Writing to Excel with prepared dataframes
with pd.ExcelWriter(excel_path) as writer:
    prepare_df(data_plasma_filtered_log_imputed_corrected).to_excel(writer, sheet_name='data_log2_imputed')
    prepare_df(data_reverted).to_excel(writer, sheet_name='data_log2')
    prepare_df(df_cli_pqtl.T).to_excel(writer, sheet_name='data_cli')

#### create covariate file

In [ ]:
data_gwas_covariates = pd.DataFrame({"intercept":[1]*data_gwas_export.shape[0]})
data_gwas_covariates.to_csv(gemma_path / 'covariates.txt', index=False, header=None)

#### Baseline characteristics

In [ ]:
para_toinclude = ['Participant ID', 'age', 'gender_num', 'bmi', 'height', 'weight', 
                   'chol', 'trigly', 'ldl', 'hdl', 
                 'glc', 'hba1c','alt', 'ast', 'ggt', 'abstinent_num', 'statin_num',
                 'kleiner', 'nas_inflam', 'nas_steatosis_ordinal','biopsy']

In [ ]:
data_cli_base=df_cli[para_toinclude].set_index('Participant ID')

In [ ]:
for col in ['nas_inflam', 'nas_steatosis_ordinal']:
    indices = data_cli_base[(data_cli_base[col].isnull()) & (data_cli_base['biopsy']=='no')].index
    data_cli_base.loc[indices, col] = 0

In [ ]:
final_included_participantIDs = df_included[df_included.index.isin(final_included_samples)]['Participant ID'].tolist()

In [ ]:
data_cli_base=data_cli_base[data_cli_base.index.isin(final_included_participantIDs)]